In [1]:
import torch
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings_model = HuggingFaceEmbeddings(
    model_name="./model/bge-base-zh-v1.5",
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
    encode_kwargs={
        "normalize_embeddings": True
    },  # 输出归一化向量，更适合余弦相似度计算
)

vectorstore = Chroma(
    embedding_function=embeddings_model,
    persist_directory="./vectorstore" #
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [2]:
question = "中国科学院国家天文台2023年部门预算总额"
sim_docs = vectorstore.similarity_search(question, k=3) #k=3， 返回最相似的3个文档

In [3]:
for doc in sim_docs:
    print(doc)

page_content='中 国 科 学 院 国家 天 文 台 2023 年 部 门 预算' metadata={'detection_class_prob': 0.46609294414520264, 'languages': '["zho", "kor"]', 'filename': 'sample.pdf', 'page_number': 1, 'element_id': 'c603b04b085efa1e14014323da2e22e5', 'source': 'knowledge_base/sample.pdf', 'category': 'Title', 'last_modified': '2026-06-29T16:32:48', 'coordinates': '{"points": [[432.126953125, 955.9337158203125], [432.126953125, 1190.3433837890625], [1254.8809814453125, 1190.3433837890625], [1254.8809814453125, 955.9337158203125]], "system": "PixelSpace", "layout_width": 1654, "layout_height": 2339}', 'file_directory': 'knowledge_base', 'filetype': 'application/pdf'}
page_content='二 、 中 国 科 学 院 国家 天 文 台 2023 年 部 门 预算' metadata={'languages': '["zho", "kor"]', 'category': 'Title', 'element_id': 'c7fbd78acdf168703339427005eb283a', 'last_modified': '2026-06-29T16:32:48', 'source': 'knowledge_base/sample.pdf', 'filename': 'sample.pdf', 'page_number': 6, 'coordinates': '{"points": [[333.208251953125, 243.46018981933

In [4]:
sim_docs = vectorstore.max_marginal_relevance_search(question, k=3)
for doc in sim_docs:
    print(doc)

page_content='中 国 科 学 院 国家 天 文 台 2023 年 部 门 预算' metadata={'last_modified': '2026-06-29T16:32:48', 'filename': 'sample.pdf', 'category': 'Title', 'coordinates': '{"points": [[432.126953125, 955.9337158203125], [432.126953125, 1190.3433837890625], [1254.8809814453125, 1190.3433837890625], [1254.8809814453125, 955.9337158203125]], "system": "PixelSpace", "layout_width": 1654, "layout_height": 2339}', 'detection_class_prob': 0.46609294414520264, 'element_id': 'c603b04b085efa1e14014323da2e22e5', 'languages': '["zho", "kor"]', 'filetype': 'application/pdf', 'file_directory': 'knowledge_base', 'source': 'knowledge_base/sample.pdf', 'page_number': 1}
page_content='中 国 科 学 院 国 家 天 文 合 2023 年 初 部 门 预算 总 额 198,223.16 万 元' metadata={'category': 'NarrativeText', 'detection_class_prob': 0.937635600566864, 'file_directory': 'knowledge_base', 'source': 'knowledge_base/sample.pdf', 'filename': 'sample.pdf', 'last_modified': '2026-06-29T16:32:48', 'languages': '["zho", "kor"]', 'parent_id': 'c7fbd78acdf

In [5]:
#获取检索器
retriever = vectorstore.as_retriever(
    search_type="similarity", #similarity 或者 mmr
    search_kwargs={"k": 3},
)
sim_docs = retriever.invoke(question)
for doc in sim_docs:
    print(doc)

page_content='中 国 科 学 院 国家 天 文 台 2023 年 部 门 预算' metadata={'file_directory': 'knowledge_base', 'source': 'knowledge_base/sample.pdf', 'filename': 'sample.pdf', 'category': 'Title', 'coordinates': '{"points": [[432.126953125, 955.9337158203125], [432.126953125, 1190.3433837890625], [1254.8809814453125, 1190.3433837890625], [1254.8809814453125, 955.9337158203125]], "system": "PixelSpace", "layout_width": 1654, "layout_height": 2339}', 'languages': '["zho", "kor"]', 'last_modified': '2026-06-29T16:32:48', 'detection_class_prob': 0.46609294414520264, 'filetype': 'application/pdf', 'element_id': 'c603b04b085efa1e14014323da2e22e5', 'page_number': 1}
page_content='二 、 中 国 科 学 院 国家 天 文 台 2023 年 部 门 预算' metadata={'page_number': 6, 'languages': '["zho", "kor"]', 'filename': 'sample.pdf', 'element_id': 'c7fbd78acdf168703339427005eb283a', 'file_directory': 'knowledge_base', 'source': 'knowledge_base/sample.pdf', 'detection_class_prob': 0.751504123210907, 'category': 'Title', 'filetype': 'applicatio

In [6]:
# claude fable5